# Pruebas de `qaoa.py` — QAOA Isla Verde sobre Quantinuum Nexus (H2)

Correr las celdas **en orden, de arriba hacia abajo** (kernel nuevo). Requiere que `qaoa.py`, `modelador_red.py`, `requirements.txt` y la carpeta `data/` (CSVs del ICE) estén en el mismo directorio que este notebook.

Notas importantes:
- `H2-Emulator` es el emulador gratis alojado en Nexus: tiene **cola compartida** (durante el hackathon un job puede tardar varios minutos en salir de `SUBMITTED`).
- Si una espera truena con `TimeoutError`, el job **sigue vivo** en Nexus: la referencia queda en `qaoa.ULTIMO_REF_JOB` y se puede volver a esperar **sin reenviar nada** (celda de rescate al final).
- La cifra reportable es siempre el **valor esperado del corte** (nunca el mejor shot, que es solo diagnóstico y está sesgado alto por construcción).

## 0 · Dependencias

In [ ]:
!pip install -r requirements.txt

## 1 · Imports y logging

Sin el `basicConfig` los logs `INFO` de httpx/qnexus quedan silenciados y no se ve el progreso de los jobs (parece colgado sin estarlo).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from pathlib import Path

import numpy as np
import qnexus as qnx

import qaoa
from qaoa import (
    cargar_instancia, construir_circuito_qaoa, valor_corte,
    obtener_proyecto_nexus, obtener_backend_local, evaluar_angulos_local,
    enviar_circuito_h2, esperar_resultado_h2, esperar_resultado_h2_paciente,
    evaluar_angulos_h2, una_ejecucion_hibrida, resolver_instancia_hibrida,
    barrer_p_hibrido, graficar_r_vs_p, guardar_resultados,
    guardar_para_comparador,
)

## 2 · Fases 1-2: autenticación y proyecto activo

Dentro del JupyterHub de Nexus `qnx.login()` es silencioso; en un entorno externo abre el flujo de login por navegador.

In [ ]:
proyecto = obtener_proyecto_nexus()
proyecto.df()

In [ ]:
# Opcional: cuotas disponibles (compilación/simulación se renuevan cada mes)
qnx.quotas.get_all().df()

## 3 · Etapa clásica: generar las instancias

Si falta la carpeta `data/` con los CSVs del ICE, `modelador_red.py` cae al proxy sintético de 8 nodos y **solo existirá `mvp8`** (`std12`/`large16` se saltan con ERROR de nodos sin resolver — es esperado en ese caso).

In [ ]:
!python modelador_red.py --data-dir data --out-dir scratch --seed 42

In [ ]:
instancia = cargar_instancia(Path("scratch"), "mvp8")
assert instancia is not None, "Falta scratch/isla_verde_mvp8.json — revisar la celda anterior"
print(f"{instancia['n']} qubits | optimo = {instancia['optimum']:.4f}")

## 4 · Paso 7.1: backend local (Qulacs) — prueba de humo

Debe correr casi instantáneo (sin red). Con ángulos aleatorios sin optimizar, `E[corte]` ronda ~40-45 sobre un óptimo de ~83.7.

In [ ]:
backend_local = obtener_backend_local()

p = 1
rng = np.random.default_rng(42)
x0 = rng.uniform(0.0, np.pi, size=2 * p)

esperado, mejor, bits = evaluar_angulos_local(x0, instancia, backend_local,
                                              shots=1000, p=p)
print(f"E[corte] local = {esperado:.4f}  (optimo = {instancia['optimum']:.4f})")

## 5 · Fases 3-5: prueba de humo en H2 (circuito de Bell, 2 qubits, 10 shots)

`enviar_circuito_h2` compila (bloqueante, rápido) y encola la ejecución **sin esperarla**: devuelve el ref del job de inmediato.

In [ ]:
from pytket import Circuit

circ_bell = Circuit(2, 2)
circ_bell.H(0)
circ_bell.CX(0, 1)
circ_bell.Measure(0, 0)
circ_bell.Measure(1, 1)

ref_bell = enviar_circuito_h2(circ_bell, n_shots=10, proyecto=proyecto)
print("Job enviado:", ref_bell.annotations.name)

In [ ]:
# Repetible las veces que se quiera: estado del job sin bloquear
print(qnx.jobs.status(ref_bell).status)

In [ ]:
# Espera paciente: un GET de status cada 30 s, hasta 30 min.
# Si truena por TimeoutError, volver a correr ESTA MISMA celda (no reenvía nada).
res_bell = esperar_resultado_h2_paciente(ref_bell, max_espera=1800, intervalo=30)
print(res_bell.get_counts())  # esperado: solo (0,0) y (1,1), nunca cruzados

## 6 · Fase 6: circuito QAOA real (mvp8, 8 qubits) en H2 + corte ponderado

In [ ]:
circ_qaoa = construir_circuito_qaoa(instancia["n"], instancia["h"],
                                    instancia["J_upper"], x0[:p], x0[p:])
ref_qaoa = enviar_circuito_h2(circ_qaoa, n_shots=1000, proyecto=proyecto)
print("Job enviado:", ref_qaoa.annotations.name)

In [ ]:
res_qaoa = esperar_resultado_h2_paciente(ref_qaoa, max_espera=1800, intervalo=30)
counts = res_qaoa.get_counts()

suma, total, mejor_corte = 0.0, 0, -np.inf
for lectura, veces in counts.items():
    bits = {instancia["variable_order"][q]: int(lectura[q])
            for q in range(instancia["n"])}
    corte = valor_corte(instancia["edges"], bits)
    suma += corte * veces
    total += veces
    mejor_corte = max(mejor_corte, corte)

print(f"E[corte] H2 = {suma / total:.4f}  (optimo = {instancia['optimum']:.4f})")
print(f"Mejor shot (solo diagnostico, no reportable) = {mejor_corte:.4f}")

## 7 · Paso 7.2: corrida híbrida completa

COBYLA optimiza los ángulos en **local** (~40 evaluaciones, segundos) y al final hace **una sola** validación en H2 (minutos de cola). `cut_h2` es la cifra reportable; debería quedar claramente por encima del ~44 sin optimizar.

In [ ]:
res_hibrida = una_ejecucion_hibrida(instancia, backend_local, proyecto,
                                    shots=1000, p=1, x0=x0)
print(f"E[corte] local (optimizado) = {res_hibrida['cut_local']:.4f}")
print(f"E[corte] H2   (reportable)  = {res_hibrida['cut_h2']:.4f}")
print(f"r = {res_hibrida['ratio_h2']:.4f}")

## 8 · Paso 7.4: estadística sobre n_runs y barrido de p

La rúbrica pide **n_runs ≥ 5**; aquí se usa 2 solo como prueba de humo porque **cada corrida = 1 job H2 en cola** (estimar varios minutos por job). Subir `n_runs` y `p_values` para la corrida final del reporte.

In [ ]:
res_stats = resolver_instancia_hibrida(instancia, backend_local, proyecto,
                                       shots=1000, p=1, n_runs=2, seed=42)
print(f"r = {res_stats['ratio_mean']:.4f} +/- {res_stats['ratio_std']:.4f} "
      f"(mejor {res_stats['ratio_best']:.4f}, peor {res_stats['ratio_worst']:.4f})")

In [ ]:
# Barrido completo (humo: 1 tier, p=1..2, n_runs=2 => 4 jobs H2).
# Para el reporte final: n_runs=5 y p_values=[1, 2, 3].
resultados = barrer_p_hibrido(Path("scratch"), ["mvp8"], [1, 2],
                              backend_local, proyecto,
                              shots=1000, n_runs=2, seed=42)

In [ ]:
# Guardar el barrido completo + el JSON PLANO que consume comparar_qaoa.py.
guardar_resultados(resultados, Path("scratch/qaoa_barrido_p.json"))
guardar_para_comparador(resultados, Path("scratch/qaoa_results.json"))

# Figura r vs p con greedy/GW MEDIDOS (scratch_dir) ademas del 0.878 teorico.
graficar_r_vs_p(resultados, Path("scratch/qaoa_ratio_vs_p.png"),
                scratch_dir=Path("scratch"))

from IPython.display import Image
Image("scratch/qaoa_ratio_vs_p.png")

## 8b · Comparación QAOA vs clásico (`comparar_qaoa.py`)

`guardar_para_comparador` emitió el JSON **plano** (`{tier: {"cut", "p", ...}}`) que espera el comparador. Esta celda genera la tabla y la figura de barras QAOA vs greedy vs GW vs óptimo.

In [ ]:
!python comparar_qaoa.py --scratch-dir scratch --tiers mvp8 --qaoa-results scratch/qaoa_results.json

from IPython.display import Image
Image("scratch/comparacion_qaoa_vs_clasico.png")

## 9 · Celda de rescate

Si cualquier espera anterior tronó con `TimeoutError`, el último job enviado sigue vivo en Nexus. Esta celda lo recupera **sin reenviar nada**.

In [ ]:
if qaoa.ULTIMO_REF_JOB is None:
    print("No hay ningún job pendiente en esta sesión")
else:
    print("Último job:", qaoa.ULTIMO_REF_JOB.annotations.name)
    print("Estado:", qnx.jobs.status(qaoa.ULTIMO_REF_JOB).status)
    # Descomentar para volver a esperarlo y bajar el resultado:
    # res = esperar_resultado_h2_paciente(qaoa.ULTIMO_REF_JOB, max_espera=1800)
    # print(res.get_counts())